In [ ]:
from amplpy import AMPL, Environment

# Initialize AMPL
ampl = AMPL()

# Read the model file
#ampl.read("/home/kkishan/Downloads/bip.mod")
ampl.read("/home/kkishan/Downloads/alan.mod")

# Get all variable entities
variables = ampl.getVariables()

# Extract integer/binary variable names for indexing
int_var_names = []
for var in ampl.getVariables():
    var_obj = var[1] if isinstance(var, tuple) else var     #If var is a tuple (key, value) → take the second element, i.e., the variable object.Else → use var directly.
    int_var_names.append(var_obj.name())


N = len(int_var_names)

# Optionally, choose a solver (if not already set in AMPL)
ampl.setOption('solver', 'cbc')  # or 'gurobi', 'cplex', etc.

# Solve the problem
ampl.solve()

# Display results
x1 = ampl.getVariable('x1').value()
x2 = ampl.getVariable('x2').value()
obj = ampl.getObjective('obj').value()
print(f"x1 = {x1}")
print(f"x2 = {x2}")
print(f"Objective = {obj}")


cbc 2.10.12:cbc 2.10.12: optimal solution; objective 5
0 simplex iterations
x1 = 0.0
x2 = 1.0
Objective = 5.0


In [16]:
# Define N (number of decision variables)
N = 5   # for example

# Generate the action set
actions = [(f"x{i+1}", direction) for i in range(N) for direction in ["L", "R"]]

# Total number of actions
num_actions = len(actions)

print("Action set:")
for a in actions:
    print(a)

print("\nTotal actions:", num_actions)


Action set:
('x1', 'L')
('x1', 'R')
('x2', 'L')
('x2', 'R')
('x3', 'L')
('x3', 'R')
('x4', 'L')
('x4', 'R')
('x5', 'L')
('x5', 'R')

Total actions: 10


In [ ]:
import itertools

# ----------------------------
# Generate valid bound vectors
# ----------------------------
def generate_valid_vectors(N):
    """Generate all 0/1 vectors with the constraint: if vec[i] == 1 then vec[N+i] must be 1"""
    for vec in itertools.product([0, 1], repeat=2 ** (2* N)):
        if all(not (vec[i] == 1 and vec[N + i] == 0) for i in range(N)):
            yield tuple(vec)





In [1]:
from amplpy import AMPL
from collections import defaultdict
import itertools
Q = defaultdict(lambda: defaultdict(float))
results = defaultdict(float)
v = defaultdict(float)
#key = (tuple(state_vector), action)
#Q[key] = updated_value
dir = ('L','R')

# --- Generate valid vectors ---
def generate_valid_vectors(N):
    """Generate all 0/1 lists with the constraint: if vec[i] == 1, then vec[N+i] must be 1."""
    for vec in itertools.product([0, 1], repeat=2 * N):
        if all(not (vec[i] == 1 and vec[N + i] == 0) for i in range(N)):
            yield list(vec)  # convert tuple → mutable list


# --- Load AMPL model once ---
ampl = AMPL()
ampl.read("/home/kkishan/Downloads/alan.mod")  # adjust path

# Extract integer/binary variable names
int_var_names = []
for var in ampl.getVariables():
    # Handle tuple (indexed variable) or direct variable
    var_obj = var[1] if isinstance(var, tuple) else var
    #if var_obj.isBinary() or var_obj.isInteger():
    int_var_names.append(var_obj.name())

# Tracking the branching decision by comparing the two bound vectors
def value_function(vec):
    """
    Update Q-values and value function v for a given state vector vec.
    vec: list of length 2*N representing current branching state.
    """
    N = len(vec) // 2  # half length
    vecL = vec.copy()  # copy for left action
    vecR = vec.copy()  # copy for right action

    for i in range(N):
        # Left action: fix variable i to 0
        vecL[i] = 0
        vecL[i + N] = 0
        # Right action: fix variable i to 1
        vecR[i] = 1
        vecR[i + N] = 1

        # Update Q-values (use tuple(vec) as key)
        Q[tuple(vec)][(i, 'L')] = abs(results.get(tuple(vec), 0.0) - results.get(tuple(vecL), 0.0)) + 0.99 * v.get(tuple(vecL), 0.0)
        Q[tuple(vec)][(i, 'R')] = abs(results.get(tuple(vec), 0.0) - results.get(tuple(vecR), 0.0)) + 0.99 * v.get(tuple(vecR), 0.0)

    # Update value function v for current vec
    v[tuple(vec)] = max(Q[tuple(vec)].values(), default=0.0)

        



N = len(int_var_names)
print("Number of integer/binary variables:", N)
#N = 4
# --- Solve relaxations efficiently ---

for episodes in range(100):
    for vec in generate_valid_vectors(N):
        for i, var_name in enumerate(int_var_names):
         var = ampl.getVariable(var_name)
         # Add both lower and upper bound constraints dynamically
         lb = vec[i]        # lower bound
         ub = vec[i + N]    # upper bound
         # Set bounds dynamically using f-string
         ampl.eval(f"{lb} <= {var_name} <= {ub};")
          
        # Solve with Gurobi
        ampl.setOption('solver', 'gurobi')
        ampl.solve()

        # Get objective value
        obj_val = ampl.getObjective('obj').value()  # replace 'obj' with your objective name
        results[vec] = obj_val
        value_function(vec)


# Print value function
print("vec".ljust(20), "Value")
print("-" * 30)
for vec, value in v.items():
    print(f"{str(vec).ljust(20)} {value}")


[ERROR] 
	line 1 offset 0
	syntax error
	context:   >>> 0  <<< <= x1 <= 0;


Number of integer/binary variables: 2


AMPLException: line 1 offset 0
syntax error
context:   >>> 0  <<< <= x1 <= 0;

For support/feedback go to https://discuss.ampl.com or e-mail <support@ampl.com>
